In [1]:
from sentence_transformers import SentenceTransformer, util

c:\Users\PC\anaconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
csim_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [3]:
def compute_cossim(context: str = None, mcq_text: str = None, model = None):
    """
    Compute QSTS score between the context and generated MCQ question.
    """
    
    # Encode the context and generated MCQ question
    context_embedding = model.encode(context, convert_to_tensor=True)
    mcq_text_embedding = model.encode(mcq_text, convert_to_tensor=True)
    
    # Calculate cosine similarity
    cosine_similarity = util.pytorch_cos_sim(context_embedding, mcq_text_embedding)
    
    return cosine_similarity.item()

In [4]:
#split context into sentences
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
def split_text_into_sentences(text):
	"""
	Split the text into sentences using NLTK.
	"""
	sentences = sent_tokenize(text)
	return sentences

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
#for each sentence, compute the cossim of the sentence and the generated MCQ question
def compute_cossim_for_sentences(context: str = None, mcq_text: str = None, model = None):
	"""
	Compute QSTS score between the context and generated MCQ question.
	"""
	sentences = split_text_into_sentences(context)
	similarity_scores = []
	for sentence in sentences:
		score = compute_cossim(context = sentence, mcq_text = mcq_text, model = model)
		similarity_scores.append(score)
	return similarity_scores

In [21]:
context = """STORM is built on a standard multimodal pipeline with key modifications for enhanced reasoning ability and token efficiency. Figure 6 depicts the detailed composition of the models. Instead of an MLP projector, STORM utilizes a linear layer followed by a Mamba-based temporal module projector to integrate spatiotemporal information into visual tokens. The architecture of STORM consists of three main components: the Mamba-Based Temporal Projector for capturing and propagating spatiotemporal information within visual tokens, the Temporal Token Compression Module for compressing temporal dimensions using training-based average pooling and/or training-free sampling, and the incorporation of a larger and more diverse dataset for improved performance across all benchmarks. STORM demonstrates consistent performance gains when trained on longer sequences, showcasing its robustness in handling various video lengths to enhance overall performance. By enabling significant token compression while preserving critical temporal information, STORM achieves state-of-the-art results on long-video understanding benchmarks and enhances computational efficiency. The implementation of STORM is based on the VILA codebase, with the introduction of the Mamba module and compression mechanisms into the architecture. Various compression methods are evaluated, with STORM variants showing superior ability to leverage long video inputs for comprehensive visual understanding. Controlled comparisons within VILA-based models highlight the advantages of Mamba-based temporal models. The efficiency of the model is analyzed, showing that STORM with temporal sampling provides improved performance with a 50% token compression ratio."""

In [22]:
respond = [['Question: What modifications were made to STORM for enhanced reasoning ability and token efficiency?',
'A. Utilization of an MLP projector',
'B. Utilization of a linear layer followed by a Mamba-based temporal module projector',
'C. Utilization of a CNN for image processing',
'D. Utilization of a transformer model',
'Answer: B. ',
'Context: STORM is built on a standard multimodal pipeline with key modifications for enhanced reasoning ability and token efficiency.'],
['Question: How many main components does the architecture of STORM consist of?',
'A. One',
'B. Two',
'C. Three',
'D. Four',
'Answer: C. ',
'Context: The architecture of STORM consists of three main components.'],
['Question: What is the purpose of the Temporal Token Compression Module in STORM?',
'A. Capturing spatiotemporal information',
'B. Compressing temporal dimensions',
'C. Incorporating a larger dataset',
'D. Enhancing visual understanding',
'Answer: B. ',
'Context: The Temporal Token Compression Module is for compressing temporal dimensions using training-based average pooling and/or training-free sampling.'],
['Question: How does STORM achieve state-of-the-art results on long-video understanding benchmarks?',
'A. By using a small dataset',
'B. By compressing tokens heavily',
'C. By enabling significant token compression while preserving critical temporal information',
'D. By relying solely on visual information',
'Answer: C. ',
'Context: STORM achieves state-of-the-art results on long-video understanding benchmarks by enabling significant token compression while preserving critical temporal information.'],
['Question: What advantages do STORM variants show in terms of leveraging long video inputs?',
'A. Ability to process only short videos',
'B. Ability to leverage long video inputs for comprehensive visual understanding',
'C. Ability to process only images',
'D. Ability to process only text',
'Answer: B. ',
'Context: STORM variants show superior ability to leverage long video inputs for comprehensive visual understanding.'],
['Question: What modifications were made to the standard multimodal pipeline in STORM to enhance reasoning ability and token efficiency?',
'A. Utilization of a linear layer',
'B. Incorporation of a MLP projector',
'C. Integration of a temporal module projector',
'D. Introduction of a spatial module projector',
'Answer: A. Utilization of a linear layer',
'Context: STORM is built on a standard multimodal pipeline with key modifications for enhanced reasoning ability and token efficiency.'],
['Question: How many main components make up the architecture of STORM?',
'A. Two',
'B. Four',
'C. Three',
'D. Five',
'Answer: C. Three',
'Context: The architecture of STORM consists of three main components: the Mamba-Based Temporal Projector, the Temporal Token Compression Module, and the incorporation of a larger and more diverse dataset.'],
['Question: What does the Temporal Token Compression Module in STORM aim to achieve?',
'A. Compression of spatial dimensions',
'B. Compression of token information',
'C. Compression of temporal dimensions',
'D. Compression of visual tokens',
'Answer: C. Compression of temporal dimensions',
'Context: The Temporal Token Compression Module aims to compress temporal dimensions using training-based average pooling and/or training-free sampling.'],
['Question: How does STORM demonstrate its robustness in handling various video lengths?',
'A. By training on shorter sequences',
'B. By achieving consistent performance gains on longer sequences',
'C. By reducing token compression',
'D. By incorporating fewer datasets',
'Answer: B. By achieving consistent performance gains on longer sequences',
'Context: STORM demonstrates consistent performance gains when trained on longer sequences, showcasing its robustness in handling various video lengths to enhance overall performance.'],
['Question: What advantage does STORM with temporal sampling provide in terms of token compression ratio?',
'A. 25%',
'B. 50%',
'C. 75%',
'D. 100%',
'Answer: B. 50%',
'Context: The efficiency of the model is analyzed, showing that STORM with temporal sampling provides improved performance with a 50% token compression ratio.']]

In [8]:
def compute_context_coverage_rate(respond, context, csim_model, top_n=4):
    """
    Computes the coverage rate of the most relevant sentences from a context 
    based on cosine similarity with MCQ questions.

    Args:
        respond (list): A list of MCQs, where each item is a list and the first element is the question.
        context (str): The input context text.
        csim_model: The cosine similarity model.
        top_n (int): Number of top similar sentences to extract per question.

    Returns:
        float: Coverage rate of the selected sentences over the original context.
    """
    # Step 1: Compute cosine similarity scores between each question and context sentences
    simscore = []
    for i in range(len(respond)):
        simscore.append(compute_cossim_for_sentences(context=context, mcq_text=respond[i][0], model=csim_model))

    # Step 2: Split context into individual sentences
    sentences = split_text_into_sentences(context)

    # Step 3: Get top-N most similar sentences for each question
    top_n_sentences = []
    for scores in simscore:
        sorted_indices = sorted(range(len(scores)), key=lambda k: scores[k], reverse=True)[:top_n]
        top_n_sentences.append([sentences[j] for j in sorted_indices])

    # Step 4: Compute coverage rate
    original_sentences = set(sentences)
    top_n_sentences_set = set(sentence for group in top_n_sentences for sentence in group)
    coverage_rate = len(top_n_sentences_set) / len(original_sentences) * 100

    return coverage_rate


In [24]:
compute_context_coverage_rate(respond, context, csim_model, top_n=6)

80.0